# 03 ? Model Training
## Predictive Hospital Readmission Risk for Chronic Disease Patients

This notebook trains reproducible, class-imbalance-aware candidate models for `readmitted_30`:

- `1`: readmitted within 30 days (`<30`)
- `0`: not readmitted within 30 days (`>30` or `NO`)

The training split is used for cross-validated hyperparameter tuning. The validation split is used only for candidate comparison and decision-threshold selection. The test split is loaded for project compatibility but is not used in any training decision.

## 1. Objective

The primary selection criteria are validation **F1**, **Recall**, **Precision**, **Average Precision**, and **ROC-AUC**, in that order. Accuracy is reported but does not dominate selection. The selected model will provide an estimated readmission risk score through `predict_proba()` and a validation-derived decision threshold.

## 2. Imports

In [ ]:
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from joblib import dump, load
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
)
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    StratifiedGroupKFold = None

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
TARGET = "readmitted_30"
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False
    print("XGBoost is not installed; the XGBoost candidate will be skipped.")

print("Libraries imported successfully.")

## 3. Load Prepared Data

Notebook 02 owns preprocessing and feature engineering. This notebook consumes its saved sparse/encoded representation without changing it. `patient_nbr` is accepted only as a grouping array when Notebook 02 has persisted one; it is never used as a feature.

In [ ]:
DATA_PATH = Path("../data/processed/model_data.pkl")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Prepared data not found at {DATA_PATH.resolve()}. "
        "Run 02_Preprocessing_Feature_Engineering.ipynb first."
    )

prepared_data = load(DATA_PATH)
X_train = prepared_data["X_train"]
X_val = prepared_data["X_val"]
X_test = prepared_data["X_test"]  # Loaded only for downstream notebook compatibility.
y_train = np.asarray(prepared_data["y_train"]).astype(int)
y_val = np.asarray(prepared_data["y_val"]).astype(int)
y_test = np.asarray(prepared_data["y_test"]).astype(int)  # Never used below.
feature_names = list(prepared_data["feature_names"])

if X_train.shape[1] != len(feature_names):
    raise ValueError("Feature count does not match feature_names.")

print(f"Training shape   : {X_train.shape}")
print(f"Validation shape : {X_val.shape}")
print(f"Test shape       : {X_test.shape}")
print(f"Encoded features : {len(feature_names)}")
print("Test labels are loaded but are not inspected or used for selection.")

## 4. Class Distribution

In [ ]:
train_class_counts = pd.Series(y_train).value_counts().sort_index()
negative_count = int(train_class_counts.get(0, 0))
positive_count = int(train_class_counts.get(1, 0))
if positive_count == 0 or negative_count == 0:
    raise ValueError("Training data must contain both target classes.")
scale_pos_weight = negative_count / positive_count
print(train_class_counts.rename(index={0: "Not readmitted", 1: "Readmitted <30 days"}))
print(f"Negative class count : {negative_count:,}")
print(f"Positive class count : {positive_count:,}")
print(f"XGBoost scale_pos_weight (training only): {scale_pos_weight:.4f}")

## 5. Baseline

In [ ]:
def metric_dict(y_true, probabilities, threshold=0.5):
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    return {
        "Accuracy": float(accuracy_score(y_true, predictions)),
        "Precision": float(precision_score(y_true, predictions, zero_division=0)),
        "Recall": float(recall_score(y_true, predictions, zero_division=0)),
        "F1 Score": float(f1_score(y_true, predictions, zero_division=0)),
        "ROC-AUC": float(roc_auc_score(y_true, probabilities)),
        "Average Precision": float(average_precision_score(y_true, probabilities)),
    }

def evaluate_fitted_model(model, X, y, threshold=0.5):
    if not hasattr(model, "predict_proba"):
        raise RuntimeError(f"{type(model).__name__} does not support predict_proba().")
    probabilities = model.predict_proba(X)[:, 1]
    return metric_dict(y, probabilities, threshold), probabilities

baseline_model = DummyClassifier(strategy="prior", random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)
baseline_metrics, baseline_val_probabilities = evaluate_fitted_model(baseline_model, X_val, y_val)
baseline_metrics["Model"] = "Dummy Baseline"
baseline_metrics["Threshold"] = 0.5
print(pd.Series(baseline_metrics))
print("The baseline has no meaningful positive-class detection when recall and F1 are zero.")

## 6. Cross-Validation Strategy

All hyperparameter searches use training data only. When persisted patient group IDs are available, the folds are `StratifiedGroupKFold`; otherwise the notebook uses `StratifiedKFold` and reports that grouped CV cannot be reconstructed without changing Notebook 02.

In [ ]:
def _first_group_array(data):
    direct_keys = [
        "patient_nbr_train", "patient_nbr_groups_train", "groups_train",
        "train_patient_nbr", "train_groups", "patient_groups_train"
    ]
    for key in direct_keys:
        if key in data:
            candidate = np.asarray(data[key])
            if len(candidate) == len(y_train):
                return candidate, key
    for key, value in data.items():
        if "patient" in key.lower() and "train" in key.lower():
            candidate = np.asarray(value)
            if len(candidate) == len(y_train):
                return candidate, key
    return None, None

groups_train, group_key = _first_group_array(prepared_data)
if groups_train is not None and StratifiedGroupKFold is not None:
    cv_strategy = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_splits = list(cv_strategy.split(X_train, y_train, groups=groups_train))
    fit_groups = groups_train
    print(f"Using StratifiedGroupKFold with patient grouping from '{group_key}'.")
elif groups_train is None:
    cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    cv_splits = list(cv_strategy.split(X_train, y_train))
    fit_groups = None
    print("No patient_nbr group IDs are present in model_data.pkl; using StratifiedKFold without inventing IDs.")
else:
    raise RuntimeError("Patient groups are present but StratifiedGroupKFold is unavailable in this sklearn version.")

SCORING = "f1"
SEARCH_N_JOBS = 1

def run_search(search, groups=None):
    start = time.perf_counter()
    if groups is None:
        search.fit(X_train, y_train)
    else:
        search.fit(X_train, y_train, groups=groups)
    return search, time.perf_counter() - start

print(f"CV folds: {len(cv_splits)}; scoring objective: {SCORING}; random_state: {RANDOM_STATE}")

## 7. Logistic Regression Tuning

In [ ]:
logistic_search = GridSearchCV(
    estimator=LogisticRegression(
        solver="liblinear", class_weight="balanced", max_iter=2000,
        random_state=RANDOM_STATE
    ),
    param_grid={"C": [0.01, 0.1, 0.5, 1, 2, 5, 10], "class_weight": ["balanced"]},
    scoring=SCORING,
    cv=cv_splits,
    n_jobs=SEARCH_N_JOBS,
    refit=True,
    return_train_score=True,
)
logistic_search, logistic_search_time = run_search(logistic_search, fit_groups)
print("Best Logistic Regression CV F1:", round(logistic_search.best_score_, 4))
print("Best parameters:", logistic_search.best_params_)

## 8. Random Forest Tuning

In [ ]:
random_forest_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1),
    param_distributions={
        "n_estimators": [200, 400, 600],
        "max_depth": [None, 10, 20],
        "min_samples_leaf": [1, 2, 5],
        "max_features": ["sqrt", "log2"],
        "class_weight": ["balanced", "balanced_subsample"],
    },
    n_iter=3,
    scoring=SCORING,
    cv=cv_splits,
    n_jobs=SEARCH_N_JOBS,
    random_state=RANDOM_STATE,
    refit=True,
    return_train_score=True,
)
random_forest_search, random_forest_search_time = run_search(random_forest_search, fit_groups)
print("Best Random Forest CV F1:", round(random_forest_search.best_score_, 4))
print("Best parameters:", random_forest_search.best_params_)

## 9. XGBoost Tuning

The search is intentionally capped to keep local execution practical while covering the requested parameter ranges. The training-only `scale_pos_weight` is fixed from the training class counts. Early stopping is not forced into cross-validation because the CV split API differs across installed XGBoost versions; the validation split remains reserved for post-search comparison and threshold selection.

In [ ]:
xgboost_search = None
xgboost_search_time = 0.0
if XGBOOST_AVAILABLE:
    xgboost_search = RandomizedSearchCV(
        estimator=XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            max_bin=64,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        param_distributions={
            "n_estimators": [200, 300, 500],
            "max_depth": [3, 4, 5, 6],
            "learning_rate": [0.03, 0.05, 0.08, 0.1],
            "min_child_weight": [1, 2, 5],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "reg_lambda": [1, 5, 10],
            "reg_alpha": [0, 0.1, 0.5],
        },
        n_iter=1,
        scoring=SCORING,
        cv=cv_splits,
        n_jobs=1,
        random_state=RANDOM_STATE,
        refit=True,
        return_train_score=True,
    )
    xgboost_search, xgboost_search_time = run_search(xgboost_search, fit_groups)
    print("Best XGBoost CV F1:", round(xgboost_search.best_score_, 4))
    print("Best parameters:", xgboost_search.best_params_)
else:
    print("XGBoost tuning skipped because xgboost is not installed.")

## 10. Model Comparison

In [ ]:
searches = {
    "Logistic Regression": (logistic_search, logistic_search_time),
    "Random Forest": (random_forest_search, random_forest_search_time),
}
if xgboost_search is not None:
    searches["XGBoost"] = (xgboost_search, xgboost_search_time)

candidate_models = {}
for model_name, (search, elapsed) in searches.items():
    candidate_models[model_name] = {
        "model": search.best_estimator_,
        "best_params": search.best_params_,
        "cv_f1": float(search.best_score_),
        "training_time_seconds": float(elapsed),
    }

print("CV search results (F1 is the tuning objective):")
cv_summary = pd.DataFrame([
    {"Model": name, "CV F1": info["cv_f1"], "Training Time": info["training_time_seconds"], "Best Hyperparameters": info["best_params"]}
    for name, info in candidate_models.items()
]).sort_values("CV F1", ascending=False)
display(cv_summary.reset_index(drop=True))

## 11. Threshold Tuning

Thresholds are selected on validation probabilities only. The fine grid is used to avoid assuming that `0.50` is appropriate for this imbalanced target. Average Precision is threshold-independent, so it is used as a documented tie-breaker after F1, Recall, and Precision.

In [ ]:
threshold_grid = np.arange(0.05, 0.76, 0.01)

def threshold_table(y_true, probabilities):
    rows = []
    average_precision = average_precision_score(y_true, probabilities)
    roc_auc = roc_auc_score(y_true, probabilities)
    for threshold in threshold_grid:
        metrics = metric_dict(y_true, probabilities, threshold)
        rows.append({
            "Threshold": float(round(threshold, 2)),
            **metrics,
        })
    return pd.DataFrame(rows), roc_auc, average_precision

for model_name, info in candidate_models.items():
    validation_table, _, _ = threshold_table(y_val, info["model"].predict_proba(X_val)[:, 1])
    best_row = validation_table.sort_values(
        ["F1 Score", "Recall", "Precision", "Average Precision", "ROC-AUC"],
        ascending=False,
    ).iloc[0]
    info["validation_probabilities"] = info["model"].predict_proba(X_val)[:, 1]
    info["threshold_table"] = validation_table
    info["best_threshold"] = float(best_row["Threshold"])
    info["default_metrics"] = metric_dict(y_val, info["validation_probabilities"], 0.50)
    info["validation_metrics"] = metric_dict(y_val, info["validation_probabilities"], info["best_threshold"])

threshold_summary = pd.DataFrame([
    {
        "Model": name,
        "Threshold": info["best_threshold"],
        **info["validation_metrics"],
    }
    for name, info in candidate_models.items()
]).sort_values(
    ["F1 Score", "Recall", "Precision", "Average Precision", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(threshold_summary)

## 12. Default vs Tuned Threshold

In [ ]:
default_vs_tuned_rows = []
for model_name, info in candidate_models.items():
    for label, threshold, metrics in [
        ("Default", 0.50, info["default_metrics"]),
        ("Tuned", info["best_threshold"], info["validation_metrics"]),
    ]:
        default_vs_tuned_rows.append({"Model": model_name, "Threshold Type": label, "Threshold": threshold, **metrics})
default_vs_tuned_df = pd.DataFrame(default_vs_tuned_rows)
display(default_vs_tuned_df[["Model", "Threshold Type", "Threshold", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC", "Average Precision"]])

## 13. Precision-Recall Analysis

In [ ]:
best_plot_name = threshold_summary.iloc[0]["Model"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for model_name, info in candidate_models.items():
    table = info["threshold_table"]
    style = "-" if model_name == best_plot_name else "--"
    axes[0, 0].plot(table["Threshold"], table["Precision"], style, label=model_name)
    axes[0, 1].plot(table["Threshold"], table["Recall"], style, label=model_name)
    axes[1, 0].plot(table["Threshold"], table["F1 Score"], style, label=model_name)
    precision, recall, _ = precision_recall_curve(y_val, info["validation_probabilities"])
    axes[1, 1].plot(recall, precision, style, label=model_name)
axes[0, 0].set(title="Precision vs Threshold", xlabel="Threshold", ylabel="Precision")
axes[0, 1].set(title="Recall vs Threshold", xlabel="Threshold", ylabel="Recall")
axes[1, 0].set(title="F1 vs Threshold", xlabel="Threshold", ylabel="F1")
axes[1, 1].set(title="Precision-Recall Curve", xlabel="Recall", ylabel="Precision")
for ax in axes.ravel():
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()
plt.show()

## 14. Overfitting Check

In [ ]:
overfitting_rows = []
for model_name, info in candidate_models.items():
    model = info["model"]
    train_probabilities = model.predict_proba(X_train)[:, 1]
    train_metrics = metric_dict(y_train, train_probabilities, info["best_threshold"])
    info["training_metrics"] = train_metrics
    gap = train_metrics["F1 Score"] - info["validation_metrics"]["F1 Score"]
    flag = "Potential overfitting" if gap > 0.20 else "No large F1 gap observed"
    overfitting_rows.append({
        "Model": model_name,
        "Training F1": train_metrics["F1 Score"],
        "Validation F1": info["validation_metrics"]["F1 Score"],
        "Training ROC-AUC": train_metrics["ROC-AUC"],
        "Validation ROC-AUC": info["validation_metrics"]["ROC-AUC"],
        "F1 Gap": gap,
        "Assessment": flag,
    })
overfitting_df = pd.DataFrame(overfitting_rows).sort_values("Validation F1", ascending=False)
display(overfitting_df)
print("Training scores are diagnostic only; selection uses validation metrics and the stated ranking.")

## 15. Final Model Selection

The final candidate is ranked transparently by tuned validation F1, Recall, Precision, Average Precision, and ROC-AUC. The Dummy Baseline is never eligible as the final model. A candidate with zero or near-zero positive-class detection is flagged and cannot win solely through accuracy.

In [ ]:
selection_rows = []
for model_name, info in candidate_models.items():
    metrics = info["validation_metrics"]
    weak_detection = metrics["Recall"] <= 1e-6 or metrics["F1 Score"] <= 1e-6
    selection_rows.append({"Model": model_name, "Weak Positive Detection": weak_detection, **metrics})
selection_df = pd.DataFrame(selection_rows).sort_values(
    ["Weak Positive Detection", "F1 Score", "Recall", "Precision", "Average Precision", "ROC-AUC"],
    ascending=[True, False, False, False, False, False],
).reset_index(drop=True)
display(selection_df)

eligible = selection_df[~selection_df["Weak Positive Detection"]]
if eligible.empty:
    raise RuntimeError("No tuned candidate demonstrates positive-class detection; refusing to select a zero-recall model.")
selected_model_name = str(eligible.iloc[0]["Model"])
selected_info = candidate_models[selected_model_name]
final_model = selected_info["model"]
selected_threshold = float(selected_info["best_threshold"])
print(f"Selected model: {selected_model_name}")
print(f"Selected validation threshold: {selected_threshold:.2f}")

## 16. Validation Prediction Examples

In [ ]:
selected_val_probabilities = final_model.predict_proba(X_val)[:, 1]
selected_val_predictions = (selected_val_probabilities >= selected_threshold).astype(int)
prediction_preview = pd.DataFrame({
    "Actual": y_val[:20],
    "Model_Estimated_Probability": selected_val_probabilities[:20].round(4),
    "Selected_Threshold": selected_threshold,
    "Predicted_Class": selected_val_predictions[:20],
    "Prediction_Label": np.where(
        selected_val_predictions[:20] == 1,
        "Readmitted within 30 days",
        "Not readmitted within 30 days",
    ),
})
display(prediction_preview)
print(
    f"Example: estimated probability={selected_val_probabilities[0]:.2f}, "
    f"threshold={selected_threshold:.2f}, prediction={selected_val_predictions[0]}"
)
print("These are model-estimated risk scores, not medically calibrated probabilities.")

## 17. Save Model

In [ ]:
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "final_model.pkl"
BUNDLE_PATH = MODEL_DIR / "model_bundle.pkl"
dump(final_model, MODEL_PATH, compress=3)
dump({"model": final_model, "threshold": selected_threshold, "feature_names": feature_names}, BUNDLE_PATH, compress=3)
print(f"Final model saved to: {MODEL_PATH.resolve()}")
print(f"Model bundle saved to: {BUNDLE_PATH.resolve()}")

## 18. Save Metadata

In [ ]:
selected_validation_metrics = selected_info["validation_metrics"]
metadata = {
    "model_name": selected_model_name,
    "selected_model": selected_model_name,
    "target": TARGET,
    "decision_threshold": selected_threshold,
    "target_definition": {
        "1": "Readmitted within 30 days",
        "0": "Not readmitted within 30 days",
    },
    "original_target_mapping": {"<30": 1, ">30": 0, "NO": 0},
    "validation_metrics": {
        "accuracy": selected_validation_metrics["Accuracy"],
        "precision": selected_validation_metrics["Precision"],
        "recall": selected_validation_metrics["Recall"],
        "f1": selected_validation_metrics["F1 Score"],
        "roc_auc": selected_validation_metrics["ROC-AUC"],
        "average_precision": selected_validation_metrics["Average Precision"],
    },
    "input_features": feature_names,
    "final_input_features": prepared_data.get("final_features", feature_names),
    "encoded_feature_count": len(feature_names),
    "best_hyperparameters": selected_info["best_params"],
    "cross_validation": {
        "strategy": "StratifiedGroupKFold" if groups_train is not None else "StratifiedKFold",
        "n_splits": 3,
        "group_key": group_key,
        "scoring": SCORING,
    },
    "random_state": RANDOM_STATE,
}
METADATA_PATH = MODEL_DIR / "model_metadata.json"
with METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4)
print(f"Metadata saved to: {METADATA_PATH.resolve()}")

## 19. Reload Verification

In [ ]:
loaded_model = load(MODEL_PATH)
loaded_probabilities = loaded_model.predict_proba(X_val)[:, 1]
loaded_predictions = (loaded_probabilities >= selected_threshold).astype(int)
if not np.array_equal(selected_val_predictions, loaded_predictions):
    raise RuntimeError("Reloaded model predictions do not match predictions before saving.")
if not np.allclose(selected_val_probabilities, loaded_probabilities, rtol=1e-10, atol=1e-12):
    raise RuntimeError("Reloaded model probabilities do not match probabilities before saving.")
print("Predictions match after reload : True")
print("Probabilities match after reload: True")
print("Saved model verification passed.")

## 20. Final Training Summary

In [ ]:
final_comparison_rows = []
for model_name, info in candidate_models.items():
    final_comparison_rows.append({
        "Model": model_name,
        "Best Hyperparameters": info["best_params"],
        "Threshold": info["best_threshold"],
        **info["validation_metrics"],
        "Training Time": info["training_time_seconds"],
    })
final_comparison_df = pd.DataFrame(final_comparison_rows).sort_values(
    ["F1 Score", "Recall", "Precision", "Average Precision", "ROC-AUC"], ascending=False
).reset_index(drop=True)
display(final_comparison_df)
print("\nSelected Model:", selected_model_name)
print("Selected Threshold:", f"{selected_threshold:.2f}")
print("Validation Accuracy:", f"{selected_validation_metrics['Accuracy']:.4f}")
print("Validation Precision:", f"{selected_validation_metrics['Precision']:.4f}")
print("Validation Recall:", f"{selected_validation_metrics['Recall']:.4f}")
print("Validation F1:", f"{selected_validation_metrics['F1 Score']:.4f}")
print("Validation ROC-AUC:", f"{selected_validation_metrics['ROC-AUC']:.4f}")
print("Validation Average Precision:", f"{selected_validation_metrics['Average Precision']:.4f}")
print("Training time:", f"{selected_info['training_time_seconds']:.2f} seconds")
print("\nTest set has not been used for model selection.")